In [15]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [16]:
# Load the clean data from the outliers notebook
df_features = pd.read_csv("../data/processed/cleaned_waitlist.csv")
df_features.head()

,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,INIT_AGE,INIT_DATE,...,DIAG_KI,MULTIORG,LISTING_CTR_CODE,outcome,event_adverse,event_transplant,censored,days_to_event,DIALYSIS_DURATION,DIALYSIS_AFTER_LISTING
0,Y,-1.0,F,B,31.63,2080.0,4099,0.0,53,2020-03-25,...,-1.0,N,13609,died,1,0,0,287.0,726.0,0
1,Y,-1.0,M,A,30.04,2070.0,4099,0.0,56,2020-02-14,...,-1.0,N,6975,removed_administrative,0,0,0,2322.0,912.0,0
2,N,-1.0,F,O,32.85,2070.0,4099,0.0,47,2020-05-27,...,-1.0,N,19716,died,1,0,0,604.0,0.0,0
3,Y,-1.0,M,A,20.00,2090.0,4099,0.0,61,2020-04-02,...,-1.0,N,8587,removed_too_sick,1,0,0,909.0,453.0,0
4,Y,-1.0,M,AB,23.30,2070.0,4010,0.0,61,2020-02-05,...,-1.0,N,18352,removed_too_sick,1,0,0,231.0,391.0,0


In [17]:
# Decoding for FUNC_STAT_TCR (per data_dictionary.md)

# Adult Karnofsky: 2010 - 2100 -> 10% - 100%
# Pediatric Lansky: 4010 - 4100 -> 10% - 100%
# Code 1.0: ADL independent -> 100%
# Codes 996, 998, -1.0, or NaN: categorical non-scale or unknown

scale_digit = df_features['FUNC_STAT_TCR'] // 1000 # Get the "scale digit" (2 or 4)

conditions = [
    scale_digit == 2,
    scale_digit == 4,
    df_features['FUNC_STAT_TCR'] == 1.0,
    df_features['FUNC_STAT_TCR'] == 996.0,
    df_features['FUNC_STAT_TCR'] == 998.0,
]

# Five labeled categories 
choices = [
    "karnofsky",
    "lansky",
    "adl_independent",
    "not_applicable_infant",
    "unknown",
]

# Sort every row into one of the five labeled categories 
df_features['functional_scale'] = np.select(conditions, choices, default="unknown")

In [18]:
# Extract functional percentage (2080 -> 80.0, 4100 -> 100.0, 1.0 -> 100.0)
is_scale = (scale_digit == 2) | (scale_digit == 4)
df_features['functional_percent'] = np.where(
    is_scale, df_features['FUNC_STAT_TCR'] % 1000, 
    np.where(df_features['FUNC_STAT_TCR'] == 1.0, 100.0, np.nan),
)

In [19]:
# Rows that don't have a real percentage (unknown / not_applicable) get funtional_percent_missing flag set to 1 
# and then filled with the median percentage to avoid missing data
df_features['functional_percent_missing'] = (df_features['functional_percent'].isna().astype(int))
df_features['functional_percent'] = df_features['functional_percent'].fillna(df_features['functional_percent'].median())

In [20]:
# Overview of functional_percent
print(df_features['functional_percent_missing'].value_counts())
df_features['functional_percent'].describe()

functional_percent_missing
0    482011
1     12792
Name: count, dtype: int64


count    494803.000000
mean         77.318347
std          14.419895
min          10.000000
25%          70.000000
50%          80.000000
75%          90.000000
max         100.000000
Name: functional_percent, dtype: float64

In [21]:
# Cap BMI at a maximum of 80, there was a data entry error
df_features.loc[df_features['BMI_TCR'] > 80, 'BMI_TCR'] = df_features['BMI_TCR'].median()

In [22]:
# Check INIT_CPRA range/outliers and most frequent values (which is 0.00)
print(df_features['INIT_CPRA'].describe())
df_features['INIT_CPRA'].value_counts().head(10)

count    494803.000000
mean          6.962676
std          21.382831
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         100.000000
Name: INIT_CPRA, dtype: float64


INIT_CPRA
0.00      420160
0.03         904
16.56        586
0.26         583
100.00       568
99.99        542
56.09        500
2.32         499
50.01        496
0.02         474
Name: count, dtype: int64

In [26]:
# Check descriptive statistics of both features
df_features['BMI_TCR'].describe()

count    494803.000000
mean         28.817576
std           5.733061
min          12.405000
25%          24.660000
50%          28.530000
75%          32.830000
max          45.085000
Name: BMI_TCR, dtype: float64

In [27]:
df_features['INIT_CPRA'].describe()

count    494803.000000
mean          6.962676
std          21.382831
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         100.000000
Name: INIT_CPRA, dtype: float64

Feature Engineering

In [28]:
# Create LISTED_POST_2021 flag marking whther a patient was listed on or after March 15, 2021
# This is the date where major kidney allocation policy took affect
init_dt = pd.to_datetime(df_features['INIT_DATE'], errors="coerce")
df_features['LISTED_POST_2021'] = (init_dt >= pd.to_datetime('2021-03-15')).astype(int)
df_features['LISTED_POST_2021'].value_counts()

LISTED_POST_2021
1    254749
0    240054
Name: count, dtype: int64

In [29]:
# Create AGE_DIALYSIS_INTERACTION feature multiplying age by years on dialysis
df_features['AGE_DIALYSIS_INTERACTION'] = df_features['INIT_AGE'] * (df_features['DIALYSIS_DURATION'] / 365.25)
df_features["AGE_DIALYSIS_INTERACTION"].describe()

count    494803.000000
mean         81.866187
std         132.761362
min           0.000000
25%           0.000000
50%          31.123888
75%         106.778919
max        2574.907598
Name: AGE_DIALYSIS_INTERACTION, dtype: float64

In [ ]:
# Log-transform to compress right-skew in DIALYSIS_DURATION
# This is a feature made to test out in modeling to scale values proportionally
df_features['DIALYSIS_DURATION_LOG'] = np.log1p(df_features['DIALYSIS_DURATION'])
df_features['DIALYSIS_DURATION_LOG'].describe()

count    494803.000000
mean          4.032315
std           3.141309
min           0.000000
25%           0.000000
50%           5.513429
75%           6.637258
max           9.639327
Name: DIALYSIS_DURATION_LOG, dtype: float64

In [31]:
# Selected the recommended features per data_dictionary.md
categorical_cols = [
    "ON_DIALYSIS",
    "GENDER",
    "ABO",
    "A2A2B_ELIGIBILITY",
    "INIT_STAT",
    "ETHCAT",
    "REGION",
    "functional_scale",
]

numerical_cols = [
    "BMI_TCR",
    "INIT_CPRA",
    "INIT_AGE",
    "functional_percent",
    "DIALYSIS_DURATION",
    "DIALYSIS_DURATION_LOG",
    "AGE_DIALYSIS_INTERACTION",
    "LISTED_POST_2021",
    "DIALYSIS_AFTER_LISTING",
    "functional_percent_missing",
]

binary_indicator_cols = [
    "LISTED_POST_2021", 
    "DIALYSIS_AFTER_LISTING", 
    "functional_percent_missing"
]

# Possible target/label features
target_cols = [
    "outcome",
    "event_adverse",
    "event_transplant",
    "censored",
    "days_to_event",
]

# Build encoded and scaled data frame
df_model = df_features[
    categorical_cols + numerical_cols + target_cols].copy()

df_model.head()

,ON_DIALYSIS,GENDER,ABO,A2A2B_ELIGIBILITY,INIT_STAT,ETHCAT,REGION,functional_scale,BMI_TCR,INIT_CPRA,...,DIALYSIS_DURATION_LOG,AGE_DIALYSIS_INTERACTION,LISTED_POST_2021,DIALYSIS_AFTER_LISTING,functional_percent_missing,outcome,event_adverse,event_transplant,censored,days_to_event
0,Y,F,B,-1.0,4099,2,8,karnofsky,31.63,0.0,...,6.588926,105.347023,0,0,0,died,1,0,0,287.0
1,Y,M,A,-1.0,4099,1,5,karnofsky,30.04,0.0,...,6.816736,139.827515,0,0,0,removed_administrative,0,0,0,2322.0
2,N,F,O,-1.0,4099,1,11,karnofsky,32.85,0.0,...,0.000000,0.000000,0,0,0,died,1,0,0,604.0
3,Y,M,A,-1.0,4099,5,7,karnofsky,20.00,0.0,...,6.118097,75.655031,0,0,0,removed_too_sick,1,0,0,909.0
4,Y,M,AB,-1.0,4010,2,7,karnofsky,23.30,0.0,...,5.971262,65.300479,0,0,0,removed_too_sick,1,0,0,231.0


In [32]:
# One-hot encode categorical data
df_encoded = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)
df_encoded.head()

,BMI_TCR,INIT_CPRA,INIT_AGE,functional_percent,DIALYSIS_DURATION,DIALYSIS_DURATION_LOG,AGE_DIALYSIS_INTERACTION,LISTED_POST_2021,DIALYSIS_AFTER_LISTING,functional_percent_missing,...,REGION_6,REGION_7,REGION_8,REGION_9,REGION_10,REGION_11,functional_scale_karnofsky,functional_scale_lansky,functional_scale_not_applicable_infant,functional_scale_unknown
0,31.63,0.0,53,80.0,726.0,6.588926,105.347023,0,0,0,...,False,False,True,False,False,False,True,False,False,False
1,30.04,0.0,56,70.0,912.0,6.816736,139.827515,0,0,0,...,False,False,False,False,False,False,True,False,False,False
2,32.85,0.0,47,70.0,0.0,0.000000,0.000000,0,0,0,...,False,False,False,False,False,True,True,False,False,False
3,20.00,0.0,61,90.0,453.0,6.118097,75.655031,0,0,0,...,False,True,False,False,False,False,True,False,False,False
4,23.30,0.0,61,70.0,391.0,5.971262,65.300479,0,0,0,...,False,True,False,False,False,False,True,False,False,False


In [33]:
# Standardize numerical data (normalize)
scaler = StandardScaler()
df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])
df_encoded[numerical_cols].describe()

,BMI_TCR,INIT_CPRA,INIT_AGE,functional_percent,DIALYSIS_DURATION,DIALYSIS_DURATION_LOG,AGE_DIALYSIS_INTERACTION,LISTED_POST_2021,DIALYSIS_AFTER_LISTING,functional_percent_missing
count,4.948030e+05,4.948030e+05,4.948030e+05,4.948030e+05,4.948030e+05,4.948030e+05,4.948030e+05,4.948030e+05,4.948030e+05,4.948030e+05
mean,9.609788e-16,2.435475e-17,1.935743e-16,-3.308283e-16,3.676189e-18,-3.676189e-18,-8.041664e-17,-2.205713e-17,2.406755e-17,-2.998392e-17
std,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00
min,-2.862798e+00,-3.256203e-01,-3.552733e+00,-4.668440e+00,-6.341516e-01,-1.283643e+00,-6.166423e-01,-1.030153e+00,-3.060295e-01,-1.629074e-01
25%,-7.251939e-01,-3.256203e-01,-6.077826e-01,-5.075179e-01,-6.341516e-01,-1.283643e+00,-6.166423e-01,-1.030153e+00,-3.060295e-01,-1.629074e-01
50%,-5.016112e-02,-3.256203e-01,1.455768e-01,1.859692e-01,-3.651158e-01,4.714961e-01,-3.822072e-01,9.707295e-01,-3.060295e-01,-1.629074e-01
75%,6.998753e-01,-3.256203e-01,7.619617e-01,8.794562e-01,1.958291e-01,8.292547e-01,1.876507e-01,9.707295e-01,-3.060295e-01,-1.629074e-01
max,2.837479e+00,4.351034e+00,2.679604e+00,1.572943e+00,1.609181e+01,1.784930e+00,1.877838e+01,9.707295e-01,3.267659e+00,6.138457e+00


In [35]:
# Verification checks
print(f"Final shape: {df_encoded.shape}")
print(
    "Total null values in features:"
    f" {df_encoded.drop(columns=target_cols).isna().sum().sum()}"
)

Final shape: (494803, 51)
Total null values in features: 0


In [36]:
# Create the encoded and scaled notebook to be downloaded
df_encoded.to_csv("../data/processed/task3_encoded_scaled.csv", index=False)

**Done:**
- Encoding + normalizing: ON_DIALYSIS, GENDER, ABO, A2A2B_ELIGIBILITY, INIT_STAT, ETHCAT, REGION, functional_scale, BMI_TCR, INIT_CPRA, INIT_AGE, DIALYSIS_DURATION, DIALYSIS_DURATION_LOG, AGE_DIALYSIS_INTERACTION

- FUNC_STAT_TCR: decoded into `functional_scale` (karnofsky/lansky/adl_independent/unknown) and `functional_percent`, with `functional_percent_missing` tracking imputed rows

- Log-transformed version of DIALYSIS_DURATION

- New features: LISTED_POST_2021 flag for the March 15, 2021 kidney transplant/allocation policy change


**Errors found:** BMI_TCR had one outlier value of 430,226 (data entry error), but is now capped at 80